In [ ]:
import numpy as np
emb = np.load("../weights/sentence_embeddings.npy")
emb.shape

In [ ]:
import matplotlib.pyplot as plt
norms = np.linalg.norm(emb,axis=1)
np.unique(norms)

In [ ]:
# variance of an embedding dimension answers how much does this dimension change across all reviews?
dim_var = emb.var(axis = 0)

# add xlabel ylabel and title of the plot
plt.figure(figsize=(15, 5))

# make the plot a bit better, like for each dimension have a point
plt.scatter(np.arange(emb.shape[1]), dim_var, color='blue', marker='o', s=10)
plt.plot(dim_var)
plt.xlabel("Embedding Dimension")
plt.ylabel("Variance")
plt.title("Variance of Embedding Dimensions")
# rotate the xticks for better readability
plt.xticks(rotation=45)
plt.xticks(np.arange(0, emb.shape[1], step=10))
plt.grid()
plt.show()

Here, in the plot, some dimensions is effectively dead, contributes nothing to distances, all values are almost same. Therefore, it carries almost no information  for distinguishing different points.

# 1. How many effective dimensions does my data really have?

In [ ]:
# compute PCA eigen values on covariance
#  and then plot eigenvalues.

from numpy.linalg import eigh

#help me compute eigen values of covariance matrix
cov = (emb.T @ emb) / emb.shape[0]
eighvals, eighvecs = eigh(cov)
plt.plot(np.flipud(eighvals)) # plot in decreasing order
plt.scatter(range(len(eighvals)), np.flipud(eighvals))
plt.xlabel("Dimension")
plt.ylabel("Eigenvalue")
plt.show()

the sharp drop above shows a low intrinsic dimensionality, and long flat tail indiscates noise. This tells us how many dimensions to keep.

In [ ]:
cum = np.cumsum(np.flipud(eighvals))/np.sum(eighvals)

# from the plot, I want to subtract the cum values by shifting one place and check if the subtraction is lower than the threshold then mark that index
diffs = np.diff(cum)
threshold = 0.01
for i, diff in enumerate(diffs):
    if diff < threshold:
        print(f"Number of principal components needed to explain variance: {i+1}")
        break

explaining_most_var_x_point = i+1

number2 = 0.7

number3 = 0.9

plt.plot(cum)
plt.axhline(y=number2, color='r', linestyle='--')
plt.axhline(y=number3, color='g', linestyle='--')
# have a horizontal line at y=f(explaining_most_var_x_point)
plt.axhline(y=cum[explaining_most_var_x_point], color='b', linestyle='--')

#Add vertical lines at the points where cumulative variance crosses 90% and 95%
idx_number1 = np.where(cum >= number2)[0][0]
idx_number3 = np.where(cum >= number3)[0][0]
plt.axvline(x=idx_number1, color='r', linestyle='--')
plt.axvline(x=idx_number3, color='g', linestyle='--')
plt.axvline(x=explaining_most_var_x_point, color='b', linestyle='--')

#mark the intersection points
plt.plot(idx_number1, cum[idx_number1], 'ro')  # red point at 90%
plt.plot(idx_number3, cum[idx_number3], 'go')  # green point at 95%
plt.plot(explaining_most_var_x_point, cum[explaining_most_var_x_point], 'bo')  # blue point at explaining_most_var_x_point

# write the x and y values at these intersection points in the plot where the lines cross
plt.text(idx_number1, 0.85, f'({idx_number1},{cum[idx_number1]:.2f})', color='r')
plt.text(idx_number3, 0.91, f'({idx_number3},{cum[idx_number3]:.2f})', color='g')
plt.text(explaining_most_var_x_point, cum[explaining_most_var_x_point]-0.05, f'({explaining_most_var_x_point},{cum[explaining_most_var_x_point]:.2f})', color='b')

plt.xlabel("Number of principal components")
plt.ylabel("Cumulative variance explained")
plt.title("Cummulative variance explained vs number of principal components")
plt.show()

In [ ]:
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import numpy as np

# ── PCA ───────────────────────────────────────────────────────────────────
X = PCA(n_components=explaining_most_var_x_point).fit_transform(emb)

# ── figure setup with explicit margins ───────────────────────────────────
fig = plt.figure(figsize=(7, 6))
fig.subplots_adjust(left=0.05, right=0.88, bottom=0.05, top=0.95)

ax = fig.add_subplot(111, projection="3d")

# ── scatter: PC4 as color, PC5 as size ───────────────────────────────────
# normalise size so points don't vanish or overwhelm
s_raw  = X[:, 4]
s_norm = 2 + 8 * (s_raw - s_raw.min()) / ((s_raw.max() - s_raw.min()) + 1e-12)

sc = ax.scatter(
    X[:, 0], X[:, 1], X[:, 2],
    c=X[:, 3], cmap="viridis",
    s=s_norm, alpha=0.6,
    depthshade=True,
)

# ── axis labels with explicit padding ────────────────────────────────────
ax.set_xlabel("PC1", fontsize=9, labelpad=14)
ax.set_ylabel("PC2", fontsize=9, labelpad=14)
ax.set_zlabel("PC3", fontsize=9, labelpad=10)

# ── shrink tick labels so they don't crowd the axis labels ───────────────
ax.tick_params(axis="both", labelsize=7, pad=2)

# ── viewing angle: all three labels stay visible ──────────────────────────
ax.view_init(elev=20, azim=-60)

# ── colorbar for PC4 outside the 3D box ──────────────────────────────────
cbar = fig.colorbar(sc, ax=ax, shrink=0.5, pad=0.12, aspect=18,
                    location="right")
cbar.set_label("PC4", fontsize=8)
cbar.ax.tick_params(labelsize=7)

# ── size legend for PC5 ───────────────────────────────────────────────────
pct = [0, 25, 50, 75, 100]
for p in pct:
    val  = np.percentile(s_raw, p)
    size = 2 + 8 * (val - s_raw.min()) / ((s_raw.max() - s_raw.min()) + 1e-12)
    ax.scatter([], [], [], s=size, c="gray", alpha=0.6,
           label=f"PC5 p{p}: {val:.2f}")
ax.legend(title="PC5 (size)", fontsize=6, title_fontsize=7,
          loc="upper left", framealpha=0.7, markerscale=1)

ax.set_title("PCA — PC1/2/3 position · PC4 colour · PC5 size",
             fontsize=9, pad=10)

plt.savefig("pca_5d.pdf", dpi=200, bbox_inches="tight")
plt.savefig("pca_5d.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# import umap
# reducer = umap.UMAP(
#     n_neighbors=15,
#     n_components=2,
#     metric='cosine',
#     random_state=42
# )
# embedding_2d = reducer.fit_transform(emb)

# plt.scatter(embedding_2d[:,0], embedding_2d[:,1], s=1, alpha=0.5)
# plt.xlabel("UMAP Dimension 1")
# plt.ylabel("UMAP Dimension 2")
# plt.title("2D UMAP projection of Review Embeddings")
# plt.show()


# 2. Pairwise Similarity Distribution Check

sample pairwise similarities

In [ ]:
# #pairwise similarity between embeddings
# from sklearn.metrics.pairwise import cosine_similarity
# sim_matrix = cosine_similarity(emb)
# sim_matrix.shape

In [ ]:
#pairwise similarity between embeddings
from sklearn.metrics.pairwise import cosine_similarity
# the above throws error, due to the huge size of the matrix
# we can compute similarity for a random sample of 5000 embeddings
size = 5000
sample_indices = np.random.choice(emb.shape[0], size=size, replace=False)
sim_matrix_sample = cosine_similarity(emb[sample_indices])
sim_matrix_sample.shape

In [ ]:
# I think sim_matrix_sample is symmetric, so I can just take the upper triangle values excluding the diagonal
upper_tri_indices = np.triu_indices_from(sim_matrix_sample, k=1)
upper_tri_values = sim_matrix_sample[upper_tri_indices]

In [ ]:
plt.hist(upper_tri_values, bins=50)
# can you draw a vertical line where the mean, median, mode is
mean_sim = np.mean(upper_tri_values)
median_sim = np.median(upper_tri_values)
ints = np.round(upper_tri_values * 100).astype(int)
vals, counts = np.unique(ints, return_counts=True)
mode_sim = vals[counts.argmax()] / 100.0
plt.axvline(x=mode_sim, color='b', linestyle='--', label=f'Mode: {mode_sim:.4f}')
plt.axvline(x=median_sim, color='g', linestyle='--', label=f'Median: {median_sim:.4f}')
plt.axvline(x=mean_sim, color='r', linestyle='--', label=f'Mean: {mean_sim:.4f}')
plt.legend()
plt.xlabel("Cosine Similarity")
plt.ylabel("Frequency")
plt.title(f"Histogram of Pairwise Cosine Similarities (Sample of {size} reviews)")
plt.show()

Positively skewed, with long positive tail, indicates a non-zero background similarity and the presence of strongly similar semantic neighbourhoods in the upper tail.

We already figured out that 16 Principal Components captures around 48% of the variance, 58 PCs around 70% of the variance and 166 PCs around 90% of the variance.

# 3. Now lets do dimensionality reduction (PCA) of the embedding space from 384 to 17, 66, 173 component space.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.preprocessing import normalize

In [ ]:
def fit_pca(emb, n_components):
    """
    Fit PCA and return transformed data + explained variance ratio.
    If emb is normalized embeddings, PCA still helps due to correlation structure.
    """
    pca = PCA(n_components=n_components, random_state=0)
    X = pca.fit_transform(emb)
    return pca, X

pca17, X17 = fit_pca(emb, 17)
pca66, X66 = fit_pca(emb, 66)
pca173, X173 = fit_pca(emb, 173)

print("Explained variance @17:", pca17.explained_variance_ratio_.sum())
print("Explained variance @66:", pca66.explained_variance_ratio_.sum())
print("Explained variance @173:", pca173.explained_variance_ratio_.sum())

In [ ]:
X17.shape, X66.shape, X173.shape

# 4. Now, we want to cluster reviews (by KMeans), finding the accurate number of clusters is important. Which we do by Silhouette score

In [ ]:
def sample_rows(X, size=20000, seed=0):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    idx = rng.choice(n, size=min(size, n), replace=False)
    return idx, X[idx]

idx_s, X17_s = sample_rows(X17, size=20000)
_, X66_s = sample_rows(X66, size=20000)
_,   X173_s = sample_rows(X173, size=20000)  # same idx not necessary, but ok either way

In [ ]:
def eval_k_grid(X, k_list, seed=0):
    """
    For each K: fit KMeans and compute silhouette, CH, DB.
    """
    rows = []
    for k in k_list:
        km = KMeans(n_clusters=k, n_init=10, random_state=seed)
        labels = km.fit_predict(X)

        # silhouette can be expensive; on 20k it's fine
        sil = silhouette_score(X, labels, metric="euclidean")
        ch  = calinski_harabasz_score(X, labels)
        db  = davies_bouldin_score(X, labels)

        rows.append({"K": k, "silhouette": sil, "calinski_harabasz": ch, "davies_bouldin": db})
    return pd.DataFrame(rows)

In [ ]:
k_list = list(range(5, 50, 5)) 
res17  = eval_k_grid(X17_s, k_list)
display(res17)

In [ ]:
def plot_k_metrics(df, title=""):
    fig, ax = plt.subplots(1, 3, figsize=(14,4))
    ax[0].plot(df["K"], df["silhouette"], marker="o")
    ax[0].set_title("Silhouette (higher better)")
    ax[0].set_xlabel("K")

    ax[1].plot(df["K"], df["calinski_harabasz"], marker="o")
    ax[1].set_title("Calinski–Harabasz (higher better)")
    ax[1].set_xlabel("K")

    ax[2].plot(df["K"], df["davies_bouldin"], marker="o")
    ax[2].set_title("Davies–Bouldin (lower better)")
    ax[2].set_xlabel("K")

    fig.suptitle(title)
    plt.show()

plot_k_metrics(res17,  "PCA(17) K selection")

In [ ]:
res66  = eval_k_grid(X66_s, k_list)
display(res66)

In [ ]:
plot_k_metrics(res66,  "PCA(66) K selection")

In [ ]:
res173 = eval_k_grid(X173_s, k_list)
display(res173)

In [ ]:
plot_k_metrics(res173,  "PCA(173) K selection")

Are the clusters that we get stable or not? Do they persist under resampling i.e. if I change the data, do I get essentially the same clusters? 

- ARI (= 1 is identical clustering, = 0 no better than random)

In [ ]:

# from sklearn.metrics import adjusted_rand_score

# def kmeans_labels(X, k, seed):
#     km = KMeans(n_clusters=k, n_init=10, random_state=seed)
#     return km.fit_predict(X)

# def stability_ari(X, k, n_trials=10, sample_size=15000, seed=0):
#     """
#     Repeatedly subsample, cluster, and compare ARI on intersection.
#     Higher ARI => more stable K.
#     """
#     rng = np.random.default_rng(seed)
#     n = X.shape[0]
#     aris = []

#     for t in range(n_trials):
#         idx1 = rng.choice(n, size=min(sample_size, n), replace=False)
#         idx2 = rng.choice(n, size=min(sample_size, n), replace=False)

#         lab1 = kmeans_labels(X[idx1], k, seed=seed+t) # Run K means with different seeds on the 2 subsets
#         lab2 = kmeans_labels(X[idx2], k, seed=seed+100+t)

#         # compare on intersection only
#         inter, i1, i2 = np.intersect1d(idx1, idx2, return_indices=True)
#         if len(inter) < 2000:
#             continue
#         aris.append(adjusted_rand_score(lab1[i1], lab2[i2]))

#     return float(np.mean(aris)) if aris else np.nan

# def stability_grid(X, k_list):
#     rows = []
#     for k in k_list:
#         ari = stability_ari(X, k)
#         rows.append({"K": k, "stability_ARI": ari})
#     return pd.DataFrame(rows)

# stab17 = stability_grid(X17_s, k_list)
# plot_k_metrics(res17.merge(stab17, on="K"), "PCA(17) + stability")  # optional
# plt.plot(stab17["K"], stab17["stability_ARI"], marker="o"); plt.title("Stability (ARI)"); plt.xlabel("K"); plt.show()

In [ ]:
best17  = res17.sort_values("silhouette", ascending=False).head(1)
best66 = res66.sort_values("silhouette", ascending=False).head(1)
best173 = res173.sort_values("silhouette", ascending=False).head(1)
print("Best PCA17:", best17.to_dict("records")[0])
print("Best PCA66:", best66.to_dict("records")[0])
print("Best PCA173:", best173.to_dict("records")[0])


In [ ]:
def distance_concentration(X, sample=5000, seed=0):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    idx = rng.choice(n, size=min(sample, n), replace=False) # randomly sample some points
    Y = X[idx].astype(np.float32)

    # compute pairwise distances efficiently using dot products:
    # ||a-b||^2 = ||a||^2 + ||b||^2 - 2 a·b
    norms = np.sum(Y*Y, axis=1, keepdims=True)
    d2 = norms + norms.T - 2*(Y @ Y.T)
    d2 = np.maximum(d2, 0.0)
    triu = d2[np.triu_indices_from(d2, k=1)]
    d = np.sqrt(triu)

    mean = d.mean()
    std  = d.std()
    cv   = std / mean
    return mean, std, cv

m17, s17, cv17 = distance_concentration(X17)
m66, s66, cv66 = distance_concentration(X66)
m173, s173, cv173 = distance_concentration(X173)
print("PCA17  dist mean/std/CV:", m17, s17, cv17)
print("PCA66  dist mean/std/CV:", m66, s66, cv66)
print("PCA173 dist mean/std/CV:", m173, s173, cv173)

# high coefficient of variation (CV) is better - distances are more spread out

In [ ]:
K = int(best17["K"].iloc[0])  # or set manually after inspecting metrics
K

From the above analysis, we get to choose 10 clusters from the reviews.

# 5. Clustering and finding top representations for each cluster and adding the aspect scores.

In [ ]:
# using PCA reduced embeddings for clustering
km = MiniBatchKMeans(n_clusters=K, batch_size=4096, random_state=0, n_init=10)
labels = km.fit_predict(X17)     # X17 is all points in PCA(17)
centers = km.cluster_centers_    # shape (K, 17)

In [ ]:
centers.shape, labels.shape

In [ ]:
# # using full embeddings for clustering
# km = MiniBatchKMeans(n_clusters=K, batch_size=4096, random_state=0, n_init=10)
# labels = km.fit_predict(emb)     # emb is all points in PCA(17)
# centers = km.cluster_centers_    # shape (K, 384)

In [ ]:
train_df = pd.read_csv("../data/train_df.csv")
train_df.head(2)

In [ ]:
import umap

def umap_plot(X, labels=None, title="UMAP", sample=30000, seed=0):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    idx = rng.choice(n, size=min(sample, n), replace=False)
    Xs = X[idx]
    ls = labels[idx] if labels is not None else None

    reducer = umap.UMAP(
        n_neighbors=15,
        n_components=2,
        metric="euclidean",
        random_state=seed
    )
    Y = reducer.fit_transform(Xs)

    plt.figure(figsize=(7,6))
    if ls is None:
        plt.scatter(Y[:,0], Y[:,1], s=2, alpha=0.5)
    else:
        plt.scatter(Y[:,0], Y[:,1], c=ls, s=2, alpha=0.6)
    plt.title(title)
    plt.show()

# UMAP on PCA(17) points (recommended for speed and denoising)
umap_plot(emb, labels=labels, title="UMAP of embeddings, colored by k-means labels")


UMAP did NOT create the clusters. It is only showing you how those clusters look when projected to 2D.
- Each dot = one review
- Colors = k-means cluster labels (found earlier in PCA space)
- Axes = UMAP coordinates (no physical meaning)

This dense, multicolored region means:
- Many reviews are semantically mixed
- They share common vocabulary and themes
- Aspects overlap (very normal in reviews)

This is expected in real text: "Reviews rarely talk about only one thing."

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import numpy as np

def pca3d_plot(X, labels, seed=0, sample=30_000, title="PCA 3D (first 3 PCs) coloured by cluster"):
    rng = np.random.default_rng(seed)
    n   = X.shape[0]
    idx = rng.choice(n, size=min(sample, n), replace=False)
    Z   = X[idx]
    ls  = labels[idx]

    fig = plt.figure(figsize=(7, 6))

    # ── leave explicit room on all sides for 3-D axis labels ──────────────
    fig.subplots_adjust(left=0.08, right=0.92, bottom=0.08, top=0.92)

    ax = fig.add_subplot(111, projection="3d")
    sc = ax.scatter(Z[:, 0], Z[:, 1], Z[:, 2],
                    c=ls, s=2, alpha=0.5, cmap="viridis")

    # ── axis labels: push them away from the tick numbers ─────────────────
    ax.set_xlabel("PC1", fontsize=9, labelpad=12)
    ax.set_ylabel("PC2", fontsize=9, labelpad=12)
    ax.set_zlabel("PC3", fontsize=9, labelpad=8)

    # ── shrink tick labels so they don't crowd the axis labels ────────────
    ax.tick_params(axis="both", labelsize=7, pad=2)

    # ── optional: tilt the view so all three labels are readable ──────────
    ax.view_init(elev=20, azim=-60)

    # ── colorbar on the right, leaving room for the z-label on the left ──
    cbar = fig.colorbar(sc, ax=ax, shrink=0.55, pad=0.1, aspect=20)
    cbar.set_label("Cluster", fontsize=8)
    cbar.ax.tick_params(labelsize=7)

    ax.set_title(title, fontsize=10, pad=10)

    plt.savefig("pca3d.pdf", dpi=200, bbox_inches="tight")
    plt.savefig("pca3d.png", dpi=200, bbox_inches="tight")
    plt.show()

In [ ]:
pca3d_plot(X17, labels)

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

def pca3d_plot(X, labels, seed=0, sample=30000):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    idx = rng.choice(n, size=min(sample, n), replace=False)
    Z = X[idx]
    ls = labels[idx]

    fig = plt.figure(figsize=(7,6))
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(Z[:,0], Z[:,1], Z[:,2], c=ls, s=2, alpha=0.5)
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_zlabel("PC3")
    plt.title("PCA 3D (first 3 PCs) colored by cluster")
    plt.show()

pca3d_plot(X17, labels)

In [ ]:
import numpy as np

def soft_assign(Z, centers, tau=1.0, chunk=20000):
    """
    Compute soft cluster membership scores S_{i,k} via
        S_{i,k} = softmax_k( -tau * ||z_i - c_k||^2 ).

    Z: (N, r), centers: (K, r)
    Returns: S: (N, K) row-stochastic (rows sum to 1).
    
    Interpretation (multiplicative tau):
      - larger tau  -> sharper / more "hard" assignments
      - smaller tau -> flatter / more uniform assignments
    """
    N = Z.shape[0]
    K = centers.shape[0]
    S = np.zeros((N, K), dtype=np.float32)

    c2 = np.sum(centers**2, axis=1)[None, :]  # (1,K)
    for start in range(0, N, chunk):
        end = min(N, start + chunk)
        Zb = Z[start:end]
        z2 = np.sum(Zb**2, axis=1, keepdims=True)            # (b,1)
        d2 = z2 + c2 - 2.0 * (Zb @ centers.T)                # (b,K)

        # ---- ONLY CHANGE IS HERE (multiplicative tau) ----
        logits = -tau * d2
        # --------------------------------------------------

        logits -= logits.max(axis=1, keepdims=True)          # stability
        expv = np.exp(logits)
        S[start:end] = expv / (expv.sum(axis=1, keepdims=True) + 1e-12)
    return S

# Example usage:
S = soft_assign(X17, centers, tau=29.5)
# S[i,k] is your soft aspect score for latent aspect k.

In [ ]:
import numpy as np

for tau in [1, 5, 10, 15, 20, 25]:
    S = soft_assign(X17, centers, tau=tau)

    row_max = S.max(axis=1)                 # how peaked each row is
    row_entropy = -(S * np.log(S + 1e-12)).sum(axis=1) / np.log(S.shape[1])  # normalized entropy in [0,1]

    print(
        f"tau={tau:>4} | "
        f"mean(max)={row_max.mean():.4f} | "
        f"p90(max)={np.quantile(row_max, 0.90):.4f} | "
        f"mean(entropy)={row_entropy.mean():.4f}"
    )

In [ ]:
import numpy as np

def squared_distances_chunked(Z, centers, chunk=20000):
    """
    Compute d2 = ||z - c||^2 in chunks without materializing NxK all at once.
    Returns: generator of (start, end, d2_chunk)
    """
    N = Z.shape[0]
    c2 = np.sum(centers**2, axis=1)[None, :]  # (1,K)
    for start in range(0, N, chunk):
        end = min(N, start + chunk)
        Zb = Z[start:end]
        z2 = np.sum(Zb**2, axis=1, keepdims=True)          # (b,1)
        d2 = z2 + c2 - 2.0 * (Zb @ centers.T)              # (b,K)
        d2 = np.maximum(d2, 0.0)                           # numerical safety
        yield start, end, d2
        2

def estimate_gap_stats(Z, centers, sample=50000, seed=0, chunk=20000):
    """
    Estimate distribution of gaps Δ = d2_2 - d2_1 using a random subset of points.
    """
    rng = np.random.default_rng(seed)
    N = Z.shape[0]
    idx = rng.choice(N, size=min(sample, N), replace=False)
    Zs = Z[idx].astype(np.float32, copy=False)

    gaps = []
    for _, _, d2 in squared_distances_chunked(Zs, centers, chunk=chunk):
        # two smallest distances per row without full sort
        two = np.partition(d2, kth=1, axis=1)[:, :2]  # (b,2) in arbitrary order
        d1 = np.min(two, axis=1)
        d2_ = np.max(two, axis=1)
        gaps.append(d2_ - d1)

    gaps = np.concatenate(gaps, axis=0)
    return {
        "gap_median": float(np.median(gaps)),
        "gap_p10": float(np.quantile(gaps, 0.10)),
        "gap_p90": float(np.quantile(gaps, 0.90)),
        "gaps": gaps,  # keep if you want to inspect histogram
    }

def choose_tau_from_gap(gap_median, r=10.0, eps=1e-12):
    """
    Multiplicative tau: target ratio r between nearest and 2nd-nearest weights.
    tau ≈ log(r) / median(gap).
    """
    return float(np.log(r) / (gap_median + eps))

stats = estimate_gap_stats(X17, centers, sample=50000, seed=0)
print(stats["gap_median"], stats["gap_p10"], stats["gap_p90"])

tau_star = choose_tau_from_gap(stats["gap_median"], r=10.0)
print("tau_star:", tau_star)

In [ ]:
S = soft_assign(X17, centers, tau=29.48)

In [ ]:
S.shape

In [ ]:
S.max(), S.min()

In [ ]:
S[0]

In [ ]:
#help me with a heatmap to checkout how the soft assignments look like
import seaborn as sns
plt.figure(figsize=(10,6))
sns.heatmap(S[:1000], cmap="viridis")
plt.title("Soft Cluster Assignments (first 1000 points)")
plt.xlabel("Cluster k")
plt.ylabel("Data point i")
plt.show()


In [ ]:
# Building a sentence level dataset.
import pandas as pd
from nltk.tokenize import sent_tokenize
from tqdm import tqdm
import re

def clean_text(x: str) -> str:
    x = re.sub(r"\s+", " ", str(x)).strip() # removes leading and trailing whitespace characters from the resulting string: cleans text
    return x


texts = train_df["review_text"].fillna("").astype(str).tolist()
bundle_of_sentences = []
for t in texts:
    text = clean_text(t)
    sents = [s.strip() for s in sent_tokenize(text)]
    sents = [s for s in sents if len(s) >= 20] # remove short sentences like okay. nice etc, short sentences cluster poorly.
    sents = sents[:10] # some reviews are extremely long, without this, long reviews may dominate clustering

    bundle_of_sentences+=sents

print("Total sentences:", len(bundle_of_sentences))

In [ ]:
bundle_of_sentences

In [ ]:
centers.shape

In [ ]:
emb.shape

In [ ]:
labels

In [ ]:
import pandas as pd

texts = bundle_of_sentences

def top_reps(Z, labels, centers, texts, k, topn=10):
    idx = np.where(labels == k)[0]
    if len(idx) == 0:
        return []
    Zk = Z[idx]
    d2 = np.sum((Zk - centers[k])**2, axis=1)
    best = idx[np.argsort(d2)[:topn]]
    return [(int(i), float(d2[np.argsort(d2)[:topn]][t]), texts[i]) for t, i in enumerate(best)]

for k in range(K):
    print("\n" + "="*90)
    print(f"LATENT ASPECT {k} | size={np.sum(labels==k)}")
    reps = top_reps(emb, labels, centers, texts, k, topn=5)
    for i, d2, txt in reps:
        cleaned = txt[:200].replace("\n", " ").replace("\r", " ")
        print(f"- d2={d2:.4f}  text={cleaned}...")
    
        # print(f"- (d2={d2:.3f}) {txt[:220].replace('\\n',' ')}...")


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def cluster_keywords(labels, texts, k, top_terms=12, max_docs=20000, seed=0):
    rng = np.random.default_rng(seed)
    idx = np.where(labels == k)[0]
    if len(idx) == 0:
        return []
    if len(idx) > max_docs:
        idx = rng.choice(idx, size=max_docs, replace=False)
    docs = [texts[i] for i in idx]

    vec = TfidfVectorizer(stop_words="english", max_features=50000, ngram_range=(1,2))
    Xtf = vec.fit_transform(docs)
    mean_tfidf = np.asarray(Xtf.mean(axis=0)).ravel()
    vocab = np.array(vec.get_feature_names_out())
    top = np.argsort(mean_tfidf)[::-1][:top_terms]
    return list(vocab[top])

for k in range(K):
    print(k, cluster_keywords(labels, texts, k))

In [ ]:
# aspect_cols = [f"asp_{k}" for k in range(K)]
# scores_df = pd.DataFrame(S, columns=aspect_cols)

# out_df = pd.concat([train_df.reset_index(drop=True), scores_df], axis=1)
# out_df["top_aspect"] = scores_df.idxmax(axis=1)
# out_df["top_aspect_score"] = scores_df.max(axis=1)

# out_df.head(2)

In [ ]:
# Building a sentence-level dataset from review-level train_df

import pandas as pd
from nltk.tokenize import sent_tokenize
from tqdm import tqdm
import re

def clean_text(x: str) -> str:
    x = re.sub(r"\s+", " ", str(x)).strip()
    return x


sentence_rows = []

for _, row in tqdm(train_df.iterrows(), total=len(train_df)):
    text = clean_text(row["review_text"])

    sents = [s.strip() for s in sent_tokenize(text)]
    sents = [s for s in sents if len(s) >= 20]
    sents = sents[:10]

    for sent_id, sent in enumerate(sents):
        new_row = row.copy()

        # Replace review_text with the sentence text
        new_row["review_text"] = sent

        # Optional but very useful tracking columns
        new_row["sentence_id"] = sent_id
        new_row["original_review_text"] = text

        sentence_rows.append(new_row)

sentence_df = pd.DataFrame(sentence_rows).reset_index(drop=True)

print("Original review-level rows:", len(train_df))
print("Sentence-level rows:", len(sentence_df))

texts = sentence_df["review_text"].fillna("").astype(str).tolist()

In [ ]:
aspect_cols = [f"asp_{k}" for k in range(K)]
scores_df = pd.DataFrame(S, columns=aspect_cols)

out_df = pd.concat(
    [sentence_df.reset_index(drop=True), scores_df.reset_index(drop=True)],
    axis=1
)

out_df["top_aspect"] = scores_df.idxmax(axis=1)
out_df["top_aspect_score"] = scores_df.max(axis=1)

out_df.head(2)

In [ ]:
train_df.shape, sentence_df.shape, scores_df.shape, out_df.shape

In [ ]:
out_df.to_csv("../data/train_with_aspect_scores_sentences.csv", index=False)